In [0]:
#Leyendo la tabla de caracteristicas generales
caso_practico_ventas = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")
# Convertir el DataFrame de Spark a Pandas
caso_practico_ventas = caso_practico_ventas.toPandas()

In [0]:
caso_practico_ventas

Exploración y Diagnóstico

In [0]:
display(caso_practico_ventas.head(10))
display(caso_practico_ventas.tail(5))

In [0]:
# deteccion de nulos
nulos_abs = caso_practico_ventas.isnull().sum()

In [0]:
# porcentaje de nulos
caso_practico_ventas.isnull().mean() * 100

In [0]:
# estadistica descritiva
caso_practico_ventas[['cantidad', 'precio_unitario']].agg(['mean', 'median', 'std', 'min', 'max'])

limpieza y depuracion

In [0]:
# tratamiento de duplicados: identificar duplicados
caso_practico_ventas.duplicated().sum()

In [0]:
# ver duplicados
caso_practico_ventas[caso_practico_ventas.duplicated()]

In [0]:
# identificar id_transaccion repetidos
caso_practico_ventas['id_transaccion'].duplicated().sum()

In [0]:
# ver transacciones repetidas
caso_practico_ventas[caso_practico_ventas['id_transaccion'].duplicated(keep=False)].sort_values('id_transaccion')

In [0]:
# eliminar duplicados
caso_practico_ventas = caso_practico_ventas.drop_duplicates()

caso_practico_ventas = caso_practico_ventas.drop_duplicates(subset=['id_transaccion'], keep='first')

normalizacion de texto 

In [0]:
# revisar metodos de pago
caso_practico_ventas['metodo_pago'].unique()

In [0]:
# normalizar metodo_pago
caso_practico_ventas['metodo_pago'] = caso_practico_ventas['metodo_pago'].replace({'Tarjeta Credito': 'Tarjeta de Crédito'})

In [0]:
# comprobar normalizacion
caso_practico_ventas['metodo_pago'].unique()

gestion de valores nulos

In [0]:
# calcular mediana de cantidad
mediana_cantidad = caso_practico_ventas['cantidad'].median()

print("Mediana de cantidad:", mediana_cantidad)

In [0]:
# reemplazar nulos de cantidad
caso_practico_ventas['cantidad'] = caso_practico_ventas['cantidad'].fillna(mediana_cantidad)

In [0]:
# comprobar nulos de cantidad
caso_practico_ventas['cantidad'].isnull().sum()

In [0]:
# calcular moda de satisfaccion 
moda_satisfaccion = caso_practico_ventas['satisfaccion_cliente'].mode()[0]

print("Moda de satisfacción:", moda_satisfaccion)

In [0]:
# reempazar nulos de satisfaccion 
caso_practico_ventas['satisfaccion_cliente'] = caso_practico_ventas['satisfaccion_cliente'].fillna(moda_satisfaccion)

In [0]:
# comprobar nulos de satisfaccion
caso_practico_ventas['satisfaccion_cliente'].isnull().sum()

tratamiento de valores atipicos 

In [0]:
# localizar el 9.999
caso_practico_ventas[caso_practico_ventas['precio_unitario'] == 9999]

In [0]:
# identificar categoria del outlier
outlier = caso_practico_ventas[caso_practico_ventas['precio_unitario'] == 9999]

categoria_outlier = outlier['categoria_producto'].iloc[0]

print("Categoría del outlier:", categoria_outlier)

In [0]:
# calcular precio promedio de la categoria
precio_promedio_categoria = caso_practico_ventas[(caso_practico_ventas['categoria_producto'] == categoria_outlier) &
(caso_practico_ventas['precio_unitario'] == 9999)]['precio_unitario'].mean()

print("Precio promedio:", precio_promedio_categoria)

In [0]:
# reemplazar el outlier
caso_practico_ventas.loc[caso_practico_ventas['precio_unitario'] == 9999,'precio_unitario'] = precio_promedio_categoria

transformacion e ingenieria de atributos


In [0]:
# columna derivada-monto_total
caso_practico_ventas['monto_total'] = (caso_practico_ventas['cantidad'] *caso_practico_ventas['precio_unitario'])

In [0]:
# comprobar monto_total
caso_practico_ventas[['cantidad', 'precio_unitario', 'monto_total']].head()

In [0]:
# convertir fecha
caso_practico_ventas['fecha'] = pd.to_datetime(caso_practico_ventas['fecha'])

In [0]:
# crear dia_semana
caso_practico_ventas['dia_semana'] = (caso_practico_ventas['fecha'].dt.day_name())

In [0]:
# comprobar fecha y dia
caso_practico_ventas[['fecha', 'dia_semana']].head()

discretizacion (binning)

In [0]:
# crear los rangos
import numpy as np
condiciones = [caso_practico_ventas['monto_total'] < 100,((caso_practico_ventas['monto_total'] >= 100) &
(caso_practico_ventas['monto_total'] <= 500)),caso_practico_ventas['monto_total'] > 500]

categorias = ['Bajo', 'Medio', 'Alto']

caso_practico_ventas['rango_monto'] = np.select(condiciones,categorias,default='Sin clasificar')

In [0]:
# comprobar los rangos
caso_practico_ventas['rango_monto'].value_counts()

agregacion y resumen de datos

In [0]:
# agrupamiento Simple (group by)
resumen_categoria = caso_practico_ventas.groupby('categoria_producto').agg(total_ingresos=('monto_total', 'sum'),promedio_unidades=('cantidad', 'mean')).reset_index()

resumen_categoria

In [0]:
# tabla dinamica (pivot table)
tabla_dinamica = pd.pivot_table(caso_practico_ventas,values='monto_total',index='region',
columns='categoria_producto',aggfunc='sum',fill_value=0)

tabla_dinamica

In [0]:
# analisis de satisfaccion 
satisfaccion_pago = caso_practico_ventas.groupby('metodo_pago')['satisfaccion_cliente'].mean().reset_index()

satisfaccion_pago

seleccion, filtrado y muestreo

In [0]:
# filtrado multicriterio
ventas_norte_satisfechos = caso_practico_ventas[(caso_practico_ventas['region'] == 'Norte') &(caso_practico_ventas['satisfaccion_cliente'] >= 4)]

ventas_norte_satisfechos

In [0]:
# numeros de registros filtrados
print("Número de ventas:",len(ventas_norte_satisfechos))

In [0]:
# muestreo estratificado
muestra_estratificada = caso_practico_ventas.groupby('region',group_keys=False).sample(frac=0.10,random_state=42)

muestra_estratificada

In [0]:
# comprobar distribucion de la muestra
muestra_estratificada['region'].value_counts()